In [ ]:
# Colab bootstrap: install requirements and stage the course data next to the notebook
import shutil
import subprocess
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    REPO_URL = "https://github.com/griu/ai-ml-finance-hands-on"
    REPO_DIR = Path("/content/ai-ml-finance-hands-on")
    NOTEBOOK_DIR = Path.cwd()
    DATA_TARGET = NOTEBOOK_DIR / "data"
    REQUIREMENTS_FILE = REPO_DIR / "requirements-colab.txt"

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_FILE)],
        check=True,
    )

    shutil.copytree(REPO_DIR / "data", DATA_TARGET, dirs_exist_ok=True)
    print(f"Colab environment detected. Data copied to: {DATA_TARGET.resolve()}")
else:
    print("Local environment detected. Colab bootstrap skipped.")


# 06c — Neural Networks and Deep Learning Awareness for Finance Students

**Course:** Data Analysis, AI and Machine Learning in Finance
**Type:** Optional self-study extension
**Instructor:** Ferran Carrascosa

---

### What this notebook covers

| Section | Topic |
|---------|-------|
| 0 | Introduction — why talk about deep learning in a finance course? |
| 1 | From a linear model to an artificial neuron |
| 2 | From one neuron to a neural network |
| 3 | Training intuition — how a network learns |
| 4 | Small tabular demo with Dataset F |
| 5 | Neural networks vs tree-based ML |
| 6 | Where deep learning may help in finance |
| 7 | Where deep learning is the wrong tool |
| 8 | Recap and self-practice |

**Dataset:** `dataset_F_deep_learning_awareness.csv` — 2 000 banking customers
(same feature schema as Dataset C, used for a compact classification demo).

### Why this matters in finance

When people hear "AI" they often think of neural networks and large language
models. This notebook puts deep learning **in perspective**:

- It is powerful — but not always necessary.
- For structured tabular data (spreadsheets, databases), tree-based models like
  XGBoost often match or beat neural networks.
- Deep learning shines with **unstructured data**: images, text, sequences.
- Understanding the basics helps you evaluate vendor claims and collaborate with
  technical teams.

---

## Section 1 — From a linear model to an artificial neuron

### Recall: logistic regression

In Notebook 05 we used logistic regression. The model computed a **weighted sum**
of the input features, added a bias, and passed the result through a sigmoid
function to produce a probability:

$$p = \sigma(w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b)$$

### An artificial neuron does the same thing

An artificial neuron (perceptron) is almost identical:

1. **Inputs** ($x_1, x_2, \ldots$) — the features.
2. **Weights** ($w_1, w_2, \ldots$) — learned importance of each feature.
3. **Bias** ($b$) — a constant offset.
4. **Activation function** — transforms the weighted sum (sigmoid, ReLU, etc.).
5. **Output** — a single number.

> **Key insight:** logistic regression *is* a single neuron with a sigmoid
> activation. Neural networks start here and add complexity by stacking many
> neurons together.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- A single neuron illustration ---
np.random.seed(42)
x = np.linspace(-5, 5, 200)

# Sigmoid activation (same as logistic regression)
sigmoid = 1 / (1 + np.exp(-x))

# ReLU activation (most common in modern networks)
relu = np.maximum(0, x)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(x, sigmoid, color="steelblue", linewidth=2)
axes[0].set_title("Sigmoid activation")
axes[0].set_xlabel("z = w·x + b")
axes[0].set_ylabel("output")
axes[0].axhline(0.5, color="grey", linestyle=":", linewidth=0.8)

axes[1].plot(x, relu, color="darkorange", linewidth=2)
axes[1].set_title("ReLU activation")
axes[1].set_xlabel("z = w·x + b")
axes[1].set_ylabel("output")

plt.tight_layout()
plt.show()

> **Sigmoid** squashes the output between 0 and 1 — useful for probabilities.
> **ReLU** (Rectified Linear Unit) outputs zero for negative inputs and passes
> positive values unchanged — it is faster to compute and is the default choice
> in most modern neural networks.

---

## Section 2 — From one neuron to a neural network

### Stacking neurons into layers

A neural network arranges neurons in **layers**:

| Layer | Role |
|-------|------|
| **Input layer** | Receives the raw features (one neuron per feature) |
| **Hidden layer(s)** | Transforms the inputs — each neuron combines outputs from the previous layer and applies an activation function |
| **Output layer** | Produces the final prediction (1 neuron for binary classification, *k* neurons for *k* classes) |

### Why hidden layers matter

A single neuron (= logistic regression) can only learn **linear boundaries**.
By adding hidden layers with non-linear activations, a neural network can learn
**complex, non-linear patterns** — the more layers and neurons, the more
expressive the model.

### A simple example

```
Input (13 features)
    ↓
Hidden layer 1 (32 neurons, ReLU)
    ↓
Hidden layer 2 (16 neurons, ReLU)
    ↓
Output (1 neuron, Sigmoid → churn probability)
```

> This is a **feed-forward network**: information flows in one direction from
> input to output. This is the simplest architecture — we will use exactly
> this pattern in the demo below.

---

## Section 3 — Training intuition: how a network learns

### The training loop

1. **Forward pass:** feed inputs through the network to get a prediction.
2. **Compute loss:** compare the prediction to the true label (e.g., binary
   cross-entropy for classification).
3. **Backward pass (backpropagation):** calculate how much each weight
   contributed to the error.
4. **Update weights:** adjust weights slightly in the direction that reduces
   the loss (gradient descent).
5. **Repeat** for many iterations (epochs).

### Key vocabulary

| Term | Plain meaning |
|------|--------------|
| **Epoch** | One full pass through the entire training set |
| **Batch** | A small chunk of training data processed at a time |
| **Learning rate** | How big each weight adjustment step is — too large and the model oscillates, too small and it learns slowly |
| **Overfitting** | The network memorises the training data instead of learning general patterns — validation loss rises while training loss drops |
| **Regularisation** | Techniques to prevent overfitting (dropout, early stopping, weight decay) |

> **You do not need to memorise the maths.** The important takeaway is that
> neural networks learn by repeatedly adjusting weights to minimise a loss
> function — the same general principle as linear regression, just with many
> more parameters.

---

## Section 4 — Small tabular demo with Dataset F

Dataset F has **2 000 banking customers** with the same features as Dataset C.
We will train a simple neural network and compare it with a logistic baseline.

We use `scikit-learn`'s `MLPClassifier` (Multi-Layer Perceptron) — it needs no
GPU and no extra libraries.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# Resolve data directory
_candidates = [Path("data/processed"), Path("../data/processed")]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Cannot locate data/processed/. "
        "Launch the notebook from the project root or the notebooks/ folder."
    )

df_f = pd.read_csv(DATA_DIR / "dataset_F_deep_learning_awareness.csv")
print(f"Dataset F: {df_f.shape[0]:,} rows × {df_f.shape[1]} columns")
print(f"Churn rate: {df_f['attrition_flag'].mean():.3f}")
df_f.head()

In [ ]:
# Encode and prepare features (same recipe as NB04)
df_enc = pd.get_dummies(df_f, columns=["geography", "gender"], drop_first=True)

target_col = "attrition_flag"
id_col = "customer_id"
y = df_enc[target_col]
X = df_enc.drop(columns=[target_col, id_col])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scale features (critical for neural networks)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Features: {X.shape[1]}")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

### Training the neural network

In [ ]:
# Simple MLP: two hidden layers (32, 16), ReLU, early stopping
mlp = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="adam",
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=42,
)
mlp.fit(X_train_sc, y_train)

print(f"Training stopped after {mlp.n_iter_} epochs (early stopping).")

### Logistic regression baseline (for comparison)

In [ ]:
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_sc, y_train)
print("Logistic regression fitted.")

### Comparing the two models

In [ ]:
results = []
for name, model in [("Logistic Regression", logreg), ("MLP Neural Network", mlp)]:
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    results.append({"Model": name, "AUC-ROC": round(auc, 3)})

comparison = pd.DataFrame(results)
print(comparison.to_string(index=False))

> On a small tabular dataset like this, the neural network and logistic
> regression typically perform **similarly**. The MLP may have a slight edge
> from capturing non-linear patterns, but the difference is usually modest.
>
> In the core notebooks, XGBoost achieved AUC ≈ 0.91 on the larger Dataset C.
> For structured tabular finance data, tree-based models remain a strong default.

---

## Section 5 — Neural networks vs tree-based ML

| Dimension | Tree-based ML (XGBoost, RF) | Neural networks (MLP, CNN, RNN) |
|-----------|---------------------------|-------------------------------|
| **Best data type** | Structured / tabular (spreadsheets, databases) | Unstructured (images, text, audio, sequences) |
| **Training speed** | Fast — minutes on a laptop | Slower — may need GPU for large models |
| **Data requirements** | Works well with hundreds to thousands of rows | Often needs tens of thousands or more |
| **Interpretability** | Feature importance is straightforward | "Black box" by default; interpretation is harder |
| **Feature engineering** | Handles raw features well; less sensitive to scaling | Requires careful scaling; benefits from feature engineering |
| **Overfitting risk** | Moderate (regularisation via depth, learning rate) | Higher — many parameters to tune; needs dropout, early stopping |
| **Finance sweet spot** | Credit scoring, churn, fraud on tabular data | Document processing, NLP, time-series with complex patterns |

> **Takeaway:** there is no universally "better" model family. The choice depends
> on the data type, the size of the dataset, and the business requirements.

---

## Section 6 — Where deep learning may help in finance

| Use case | Data type | Why deep learning helps |
|----------|-----------|----------------------|
| **Sentiment analysis** | News headlines, earnings calls, social media | Understands language context and nuance |
| **Document extraction** | Contracts, invoices, regulatory filings | Reads unstructured documents at scale |
| **Fraud detection on sequences** | Transaction streams over time | Captures sequential patterns that tabular models miss |
| **Alternative data** | Satellite images, web traffic, app usage | Processes non-tabular signals |
| **Time-series forecasting** | High-frequency or complex temporal data | Recurrent and transformer architectures model long dependencies |

> These are areas where the **data is naturally unstructured or sequential** —
> exactly where neural networks add value beyond what XGBoost can offer.

---

## Section 7 — Where deep learning is the wrong tool

| Situation | Why a simpler model is better |
|-----------|------------------------------|
| **Small dataset** (< 5 000 rows) | Neural networks need more data to generalise; tree models work well with less |
| **High interpretability required** | Regulators and auditors often need feature-level explanations that trees provide naturally |
| **Limited compute** | Training and serving deep models costs more in time and infrastructure |
| **Marginal improvement** | If XGBoost already achieves AUC 0.91, a neural network reaching 0.92 may not justify the added complexity |
| **Tabular-only data** | On structured tables, boosted trees match or beat neural networks in most benchmarks |

> **A good rule of thumb for finance Master's students:** start with simple
> baselines (logistic regression → trees → XGBoost). Move to neural networks
> only when the data or the problem **specifically calls for it**.

---

## Section 8 — Recap

### What to remember

| Concept | Key takeaway |
|---------|-------------|
| **Artificial neuron** | Weighted sum + bias + activation — logistic regression is a single neuron |
| **Neural network** | Multiple layers of neurons that learn non-linear patterns |
| **Training** | Forward pass → loss → backpropagation → weight update, repeated over many epochs |
| **Scaling** | Neural networks are sensitive to feature scale — always standardise |
| **vs Tree-based ML** | Trees excel on tabular data; neural networks excel on unstructured data |
| **Finance context** | Start simple; use deep learning when the problem (not the hype) demands it |

---

### Self-practice

1. In the demo above, change the hidden layer sizes to `(64, 32, 16)`. Does
   the AUC improve? Does training take longer?
2. Remove early stopping (`early_stopping=False`) and set `max_iter=500`.
   What happens to training — does it converge or overfit?
3. Pick one finance use case (e.g., fraud detection) and write 3 sentences
   explaining whether you would start with XGBoost or a neural network, and why.
4. Explain in your own words why scaling is more important for neural networks
   than for decision trees.

---

*End of Notebook 06c — Neural Networks and Deep Learning Awareness.*